In [4]:
!pip install -q torch transformers ninja tokenizers rwkv pynvml huggingface_hub smolagents

In [75]:
!pip install -q ddgs wikipedia-api

  Preparing metadata (setup.py) ... done


In [76]:
import os
import copy
import time
import gc

import torch
import huggingface_hub
from transformers import AutoTokenizer

from smolagents.agents import EMPTY_PROMPT_TEMPLATES
from smolagents.models import Model, ChatMessage, MessageRole
from smolagents.monitoring import TokenUsage
from smolagents.tools import Tool
from smolagents.default_tools import (
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    VisitWebpageTool,
    WikipediaSearchTool
)
from smolagents import CodeAgent

from IPython.display import clear_output

os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

from rwkv.model import RWKV
from rwkv.utils import PIPELINE, PIPELINE_ARGS

In [7]:
model_title = model_title = "rwkv7-g1a-2.9b-20250924-ctx4096"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")
rwkv_model = RWKV(model=model_path.replace('.pth',''), strategy='cuda fp16')
rwkv_pipeline = PIPELINE(rwkv_model, "rwkv_vocab_v20230424")

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/71df75d4c929166e220998f32bdf3139e54c36b8/rwkv7-g1a-2.9b-20250924-ctx4096 (cuda fp16)



In [41]:
class RWKVModel(Model):
    """
    A subclass of Model for RWKV7 inference, adapted from the experimental notebook.
    """
    def __init__(
        self,
        model,
        pipeline,
        flatten_messages_as_text: bool = False,
        tool_name_key: str = "name",
        tool_arguments_key: str = "arguments",
        model_id: str | None = None,
        **kwargs,
    ):    
        # Setup RWKV
        self.CTX_LIMIT = 1000
        self.PENALTY_DECAY = 0.996
        self._model = model
        self._pipeline = pipeline
        
        # Only for chat template
        self.tokenizer = AutoTokenizer.from_pretrained('fla-hub/rwkv7-2.9B-g1', trust_remote_code=True)
        
        super().__init__(flatten_messages_as_text=True, model_id="model_id", **kwargs)

    def generate(
        self,
        messages: list[ChatMessage],
        stop_sequences: list[str] | None = None,
        response_format: dict[str, str] | None = None,
        tools_to_call_from: list[Tool] | None = None,
        **kwargs,
    ) -> ChatMessage:
        """Process the input messages and return the model's response."""
        completion_kwargs = self._prepare_completion_kwargs(
            messages=messages,
            flatten_messages_as_text=self.flatten_messages_as_text,
            stop_sequences=stop_sequences,
            tools_to_call_from=tools_to_call_from,
            **kwargs,
        )

        print("Completion kwargs", completion_kwargs)

        messages = completion_kwargs.pop("messages")
        prepared_stop_sequences = completion_kwargs.pop("stop", [])
        tools = completion_kwargs.pop("tools", None)
        completion_kwargs.pop("tool_choice", None)

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            tools=tools,
            add_generation_prompt=True,
            enable_thinking=False
        )

        # Remove <|rwkv_tokenizer_end_of_text|>
        prompt.replace("<|rwkv_tokenizer_end_of_text|>", "")

        print(f"Input prompt: ", prompt)

        output = self.run_rwkv(prompt)

        output_text = output.strip()

        print(f"Output: ", output_text)
        
        return ChatMessage(
            role=MessageRole.ASSISTANT,
            content=output_text,
            raw={"out": output_text},
            token_usage=TokenUsage(
                input_tokens=len(prompt.split()),
                output_tokens=len(output_text.split()),
            ),
        )

    def run_rwkv(
        self,
        ctx,
        token_count=200,
        temperature=1.0,
        top_p=0.7,
        presencePenalty=0.1,
        countPenalty=0.1,
    ):
        args = PIPELINE_ARGS(
            temperature=max(0.2, float(temperature)),
            top_p=float(top_p),
            alpha_frequency=countPenalty,
            alpha_presence=presencePenalty,
            token_ban=[],        # ban the generation of some tokens
            token_stop=[0]       # stop generation whenever you see any token here
        )
    
        ctx = ctx.strip()
        all_tokens = []
        out_last = 0
        out_str = ''
        occurrence = {}
        state = None
    
        # Start timing
        start_time = time.time()
    
        for i in range(int(token_count)):
            input_ids = self._pipeline.encode(ctx)[-self.CTX_LIMIT:] if i == 0 else [token]
            out, state = self._model.forward(input_ids, state)
    
            for n in occurrence:
                out[n] -= (args.alpha_presence + occurrence[n] * args.alpha_frequency)
    
            token = self._pipeline.sample_logits(
                out,
                temperature=args.temperature,
                top_p=args.top_p)
            if token in args.token_stop:
                break
    
            all_tokens.append(token)
    
            for xxx in occurrence:
                occurrence[xxx] *= self.PENALTY_DECAY
    
            ttt = self._pipeline.decode([token])
            www = 1
            if ttt in ' \t0123456789':
                www = 0
            if token not in occurrence:
                occurrence[token] = www
            else:
                occurrence[token] += www
    
            tmp = self._pipeline.decode(all_tokens[out_last:])
            if '\ufffd' not in tmp:
                out_str += tmp
                out_last = i + 1

                if out_str.strip().endswith("</code>"):
                    break
    
                if out_str.strip().endswith("Observation:"):
                    break
    
        # Cleanup and timing
        del out
        del state
        gc.collect()
        torch.cuda.empty_cache()
        
        return out_str.strip()

In [42]:
model = RWKVModel(model=rwkv_model, pipeline=rwkv_pipeline)

In [43]:
model.run_rwkv("Hello?")

'Peter, it\'s Charlie." \n\n"Hi, Charlie. How are you doing?" \n\n"I\'m fine, and how are you?" \n\n"I\'m great, actually. I\'m on my way home from work now." \n\n"Really? What do you do?" \n\n"I\'m a lawyer. I just closed a case." \n\n"That\'s great! I\'m so happy for you!" \n\n"Thanks. So, what are you up to tonight?" \n\n"Well, I was thinking of going out with some friends, but I\'m not sure if \nthey\'ll be able to make it." \n\n"Oh, okay. Well, maybe you could come over to my place instead?" \n\n"Sure, that sounds great! I\'ll see you soon." \n\nThey hung up and Peter put the phone down on the counter. He took a deep \nbreath and tried to calm himself down. He had been wanting to see Charlie for \na while now, but he was worried'

In [65]:
SYSTEM_PROMPT = '''System: You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '{{code_block_opening_tag}}', and closed with '{{code_block_closing_tag}}'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.
Here are a few examples using notional tools:
---
User: New task:
What is the result of the following operation: 5 + 3 + 1294.678?

Assistant:<think>
</think>Thought: I will use Python code to compute the result of the operation and then return the final answer using the `final_answer` tool.
{{code_block_opening_tag}}
result = 5 + 3 + 1294.678
final_answer(result)
{{code_block_closing_tag}}
---
User: New task:
"What is the current age of the pope, raised to the power 0.36?"

Assistant:<think>
</think>Thought: I will use the tool `wikipedia_search` to get the age of the pope, and confirm that with a web search.
{{code_block_opening_tag}}
pope_age_wiki = wikipedia_search(query="current pope age")
print("Pope age as per wikipedia:", pope_age_wiki)
pope_age_search = web_search(query="current pope age")
print("Pope age as per google search:", pope_age_search)
{{code_block_closing_tag}}
Observation:
Pope age: "The pope Francis is currently 88 years old."
Thought: I know that the pope is 88 years old. Let's compute the result using Python code.
{{code_block_opening_tag}}
pope_current_age = 88 ** 0.36
final_answer(pope_current_age)
{{code_block_closing_tag}}
---
Above examples were using notional tools that might not exist for you.
On top of performing computations in the Python code snippets that you create, you only have access to these tools, behaving like regular python functions:
{{code_block_opening_tag}}
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{%- endfor %}
{{code_block_closing_tag}}
Here are the rules you should always follow to solve your task:
1. Always provide a 'Thought:' sequence, and a '{{code_block_opening_tag}}' sequence ending with '{{code_block_closing_tag}}', else you will fail.
2. Use only variables that you have defined!
3. Always use the right arguments for the tools. DO NOT pass the arguments as a dict as in 'answer = wikipedia_search({'query': "What is the place where James Bond lives?"})', but use the arguments directly as in 'answer = wikipedia_search(query="What is the place where James Bond lives?")'.
4. Call a tool only when needed, and never re-do a tool call that you previously did with the exact same parameters.
5. Don't name any new variable with the same name as a tool: for instance don't name a variable 'final_answer'.
6. Never create any notional variables in our code, as having these in your logs will derail you from the true variables.
7. You can use imports in your code, but only from the following list of modules: {{authorized_imports}}
8. The state persists between code executions: so if in one step you've created variables or imported modules, these will all persist.
9. Don't give up! You're in charge of solving the task, not providing directions to solve it.
Now Begin!'''

In [66]:
RWKV_PROMPT_TEMPLATES = EMPTY_PROMPT_TEMPLATES
RWKV_PROMPT_TEMPLATES["system_prompt"] = SYSTEM_PROMPT

In [77]:
agent = CodeAgent(
    tools=[FinalAnswerTool(), DuckDuckGoSearchTool(), VisitWebpageTool(), WikipediaSearchTool()],
    model=model,
    prompt_templates=RWKV_PROMPT_TEMPLATES,
    verbosity_level=2)

In [78]:
agent.run("What is the result of 2 + exp(10)?", max_steps=1)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the result of 2 + exp(10)?                                                                              │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Completion kwargs {'messages': [{'role': <MessageRole.SYSTEM: 'system'>, 'content': 'System: You are an expert assistant who can solve any task using code blobs.\nYou will be given a task to solve as best you can.\nTo do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.\nTo solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.\nAt each step, in the \'Thought:\' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.\nThen in the Code sequence you should write the code in simple Python. The code sequence must be opened with \'<code>\', and closed with \'</code>\'.\nDuring each intermediate step, you can use \'print()\' to save whatever important information you will then need.\nThese print outputs will then appear in the \'Observation:\' field, which will be available as input for the nex

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will use the `math` module to compute the value of exp(10) and then add it to 2.                        
<code>                                                                                                             
import math                                                                                                        
result = 2 + math.exp(10)                                                                                          
print(result)                                                                                                      
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import math                                                                                                      
  result = 2 + math.exp(10)                                                                                        
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

[Step 1: Duration 3.36 seconds| Input tokens: 727 | Output tokens: 29]

Completion kwargs {'messages': [{'role': <MessageRole.SYSTEM: 'system'>, 'content': ''}, {'role': <MessageRole.USER: 'user'>, 'content': 'New task:\nWhat is the result of 2 + exp(10)?'}, {'role': <MessageRole.ASSISTANT: 'assistant'>, 'content': "Thought: I will use the `math` module to compute the value of exp(10) and then add it to 2.\n<code>\nimport math\nresult = 2 + math.exp(10)\nprint(result)\n</code>\nCalling tools:\n[{'id': 'call_1', 'type': 'function', 'function': {'name': 'python_interpreter', 'arguments': 'import math\\nresult = 2 + math.exp(10)\\nprint(result)'}}]"}, {'role': <MessageRole.USER: 'user'>, 'content': 'Observation:\nExecution logs:\n22028.465794806718\nLast output from code snippet:\nNone\n'}]}
Input prompt:  <|rwkv_tokenizer_end_of_text|>System: 

User: New task:
What is the result of 2 + exp(10)?

Assistant: Thought: I will use the `math` module to compute the value of exp(10) and then add it to 2.
<code>
import math
result = 2 + math.exp(10)
print(result)
</c

Reached max steps.

[Step 2: Duration 1.62 seconds| Input tokens: 799 | Output tokens: 37]

'The result of 2 + exp(10) is 22028.465794806718.'

In [79]:
agent.run("What is the most popular programming language?", max_steps=1)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the most popular programming language?                                                                  │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Completion kwargs {'messages': [{'role': <MessageRole.SYSTEM: 'system'>, 'content': 'System: You are an expert assistant who can solve any task using code blobs.\nYou will be given a task to solve as best you can.\nTo do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.\nTo solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.\nAt each step, in the \'Thought:\' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.\nThen in the Code sequence you should write the code in simple Python. The code sequence must be opened with \'<code>\', and closed with \'</code>\'.\nDuring each intermediate step, you can use \'print()\' to save whatever important information you will then need.\nThese print outputs will then appear in the \'Observation:\' field, which will be available as input for the nex

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to search for the most popular programming language to determine the answer. I'll use the          
web_search tool for this purpose.                                                                                  
<code>                                                                                                             
web_search(query="most popular programming language")                                                              
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  web_search(query="most popular programming language")                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

[Step 1: Duration 3.60 seconds| Input tokens: 726 | Output tokens: 29]

Completion kwargs {'messages': [{'role': <MessageRole.SYSTEM: 'system'>, 'content': ''}, {'role': <MessageRole.USER: 'user'>, 'content': 'New task:\nWhat is the most popular programming language?'}, {'role': <MessageRole.ASSISTANT: 'assistant'>, 'content': 'Thought: I need to search for the most popular programming language to determine the answer. I\'ll use the web_search tool for this purpose.\n<code>\nweb_search(query="most popular programming language")\n</code>\nCalling tools:\n[{\'id\': \'call_1\', \'type\': \'function\', \'function\': {\'name\': \'python_interpreter\', \'arguments\': \'web_search(query="most popular programming language")\'}}]'}, {'role': <MessageRole.USER: 'user'>, 'content': "Observation:\nExecution logs:\nLast output from code snippet:\n## Search Results\n\n[Wikipedia Programming languages used in most popular websites - Wikipedia](https://en.wikipedia.org/wiki/Programming_languages_used_in_most_popular_websites)\nSeptember 9, 2025 - One thing the most visite

Reached max steps.

[Step 2: Duration 6.52 seconds| Input tokens: 1,431 | Output tokens: 107]

"Based on the provided text, the most popular programming languages in 2024 (as of the article's publication date) are:\n1. **JavaScript**  \n2. **HTML/CSS**  \n3. **Python**  \n4. **C++**  \n5. **Java**  \n6. **C#**  \n7. **Kotlin**  \n8. **Swift**  \n9. **PHP**  \n10. **Ruby**  \n11. **Go**  \n12. **Rust**  \nThese rankings are based on various sources, including TIOBE Index, GitHub Octoverse, and Statista, which collectively highlight JavaScript and Python as the most widely used languages across industries like web development, AI, and general-purpose programming."